# ZotVision — CNN-ViT Hybrid Training (TPU)
Runtime → Change runtime type → TPU before running.

In [ ]:
!pip install -q efficientnet_pytorch transformers
!git clone https://github.com/varshinivij/ZotVision.git
!git -C /content/ZotVision checkout hoon
import sys
sys.path.append('/content/ZotVision/zot-vision/backend')

In [ ]:
import sys
sys.path = [p for p in sys.path if 'ZotVision' not in p]
from google.colab import drive
sys.path.append('/content/ZotVision/zot-vision/backend')
drive.mount('/content/drive')

import os

IMAGES_DIR  = '/content/ZotVision/zot-vision/datasets/images'
LABELS_FILE = '/content/ZotVision/zot-vision/datasets/results/labels.txt'
WEIGHTS_DIR = '/content/drive/MyDrive/zotvision_weights'
os.makedirs(WEIGHTS_DIR, exist_ok=True)
print(f'Images: {len(os.listdir(IMAGES_DIR))} files')
print(f'Weights will save to: {WEIGHTS_DIR}')

In [ ]:
import torch
import torch_xla
import torch_xla.core.xla_model as xm

DEVICE = xm.xla_device()
print(f'Device: {DEVICE}')

In [ ]:
import transformer
transformer.IMAGES_DIR  = IMAGES_DIR
transformer.LABELS_FILE = LABELS_FILE
transformer.DEVICE      = DEVICE

from transformer import (
    CNNViTHybrid, CustomDataset, load_samples, get_transforms,
    LABEL_MAP, NUM_CLASSES, BATCH_SIZE, NUM_EPOCHS, LR
)

import random
import torch.nn as nn
from torch.utils.data import DataLoader

all_samples = load_samples(LABELS_FILE, IMAGES_DIR)
random.shuffle(all_samples)
split = int(0.8 * len(all_samples))
train_ds = CustomDataset(all_samples[:split], get_transforms(True))
val_ds   = CustomDataset(all_samples[split:],  get_transforms(False))
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
print(f'Train: {len(train_ds)} | Val: {len(val_ds)}')

model = CNNViTHybrid(dropout=0.3).to(DEVICE)
print(f'Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

In [ ]:
# ── Hyperparameter overrides ──
BATCH_SIZE = 8
NUM_EPOCHS = 30
LR         = 1e-4
PATIENCE   = 7

# Rebuild loaders with updated BATCH_SIZE
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
print(f'BATCH_SIZE={BATCH_SIZE}  NUM_EPOCHS={NUM_EPOCHS}  LR={LR}  PATIENCE={PATIENCE}')

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
WEIGHTS_PATH = os.path.join(WEIGHTS_DIR, 'model_weights.pth')
best_val_acc = 0.0
no_improve   = 0

history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    tl, correct, total = 0.0, 0, 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        logits = model(imgs)
        loss   = criterion(logits, labels)
        loss.backward()
        xm.optimizer_step(optimizer)
        xm.mark_step()
        tl      += loss.item() * imgs.size(0)
        correct += (logits.argmax(1) == labels).sum().item()
        total   += imgs.size(0)
    train_loss, train_acc = tl/total, correct/total

    model.eval()
    tl, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            logits = model(imgs)
            tl      += criterion(logits, labels).item() * imgs.size(0)
            correct += (logits.argmax(1) == labels).sum().item()
            total   += imgs.size(0)
            xm.mark_step()
    val_loss, val_acc = tl/total, correct/total
    scheduler.step()

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    print(f'Epoch {epoch:03d}/{NUM_EPOCHS} | Train Loss: {train_loss:.4f}  Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f}  Acc: {val_acc:.4f}')

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        no_improve   = 0
        xm.save(model.state_dict(), WEIGHTS_PATH)
        print(f'  Saved best model (val_acc={best_val_acc:.4f}) -> {WEIGHTS_PATH}')
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f'  Early stopping at epoch {epoch} (no improvement for {PATIENCE} epochs)')
            break

print(f'Training complete. Best Val Acc: {best_val_acc:.4f}')

In [ ]:
import matplotlib.pyplot as plt

epochs = range(1, len(history['train_loss']) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(epochs, history['train_loss'], label='Train Loss')
ax1.plot(epochs, history['val_loss'],   label='Val Loss')
ax1.set_title('Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(True)

ax2.plot(epochs, history['train_acc'], label='Train Acc')
ax2.plot(epochs, history['val_acc'],   label='Val Acc')
ax2.set_title('Accuracy')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.legend()
ax2.grid(True)

plt.suptitle('ZotVision Training', fontsize=14)
plt.tight_layout()

plot_path = os.path.join(WEIGHTS_DIR, 'training_curves.png')
plt.savefig(plot_path, dpi=150)
plt.show()
print(f'Plot saved to {plot_path}')